<a href="https://colab.research.google.com/github/anacvetkoo/MOS-projekt/blob/main/notebooks/Ana_priprava_podatkov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Priprava podatkovnih zbirk za analizo dostopnosti, zaključevanja študija, stroškov in diplomantov računalništva na osmih univerzah Ivy League.

Vir podatkov: Integrated Postsecondary Education Data System (IPEDS), National Center for Education Statistics (NCES).

Povezava: https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx?gotoReportId=7&fromIpeds=true&sid=a7f02a68-5be1-4416-949f-a6c3e236838d&rtid=7


In [ ]:

import requests #za prenos datotek s spleta
import zipfile #za odpiranje zip-ov
import io #branje prenesenih datotek iz pomnilnika
import pandas as pd

leta = range(2004, 2025)
ivy = {
    217156: "Brown", #ivy univerze po uradnem id-ju UNITID
    190150: "Columbia",
    190415: "Cornell",
    182670: "Dartmouth",
    166027: "Harvard",
    215062: "Penn",
    186131: "Princeton",
    130794: "Yale"
}

def prenesi_datoteko(ime): #prenese letno zip-datoteko iz IPEDS, vrne kot df
    povezave = [
        f"https://nces.ed.gov/ipeds/datacenter/data/{ime}.zip",
        f"https://nces.ed.gov/ipeds/complete-data-files/{ime}.zip" #datoteka za 2024 je na tej povezavi
    ]

    zadnja_napaka = None

    for url in povezave:
        try:
          r = requests.get(url, timeout=60) #prenos zip-a
          r.raise_for_status()

          with zipfile.ZipFile(io.BytesIO(r.content)) as z: #odpiranje zip-a direkt iz pomnilnika
              csv = next( #iskanje prve .csv datoteke
                  (f for f in z.namelist()
                  if f.lower().endswith(".csv")),
                  None
              )
              if csv is None:
                  raise ValueError("CSV ni v ZIP datoteki")

              with z.open(csv) as f: #csv prebere v tabelo
                  return pd.read_csv(
                      f, encoding="latin1", low_memory=False
                  )

        except (requests.RequestException,
                zipfile.BadZipFile, ValueError) as e:
            zadnja_napaka = e

    raise RuntimeError(
        f"Prenos datoteke {ime} ni uspel: {zadnja_napaka}"
    )




RV1: Kako so se spreminjali prijave, sprejemi in vpisi na univerzah Ivy League?

Za obdobje 2004–2024 sem pridobila podatke o:
- številu prijavljenih kandidatov(APPLCN)
- sprejetih kandidatov (ADMSSN)
- dejansko vpisanih kandidatov (ENRLT)

Za starejša leta (do 2013): datoteke Institutional Characteristics (IC)<br><br>
za novejša leta(od 2014): datoteke Admissions (ADM).

In [ ]:
rezultati = []
napake = [] #seznam let pri katerih pridobivanje podatkov ni uspelo

for leto in leta:
    kandidati = (
        [f"IC{leto}", f"ADM{leto}"]
        if leto <= 2013
        else [f"ADM{leto}", f"IC{leto}"]
    )

    uspesno = False
    zadnja_napaka = "Ustrezni podatki niso bili najdeni."

    for ime in kandidati:
        try:
            df = prenesi_datoteko(ime)

            zahtevani = [ #stolpci ki jih rabim za RV1
                "UNITID", "APPLCN", "ADMSSN", "ENRLT"
            ]

            if not all(c in df.columns for c in zahtevani):
                continue

            df = df[zahtevani].copy()

            df["leto"] = leto
            df["vir"] = ime
            df["ivy_league"] = df["UNITID"].isin(ivy) #stolpec ce je ivy league ali ne
            df["univerza"] = df["UNITID"].map(ivy) #imena univerz za ivy league

            rezultati.append(df)
            print(f"OK: {leto} ({ime}), {len(df)} univerz")
            uspesno = True
            break

        except Exception as e:
            zadnja_napaka = str(e)
#belezenje podatkov, ki jih nisem dobila
    if not uspesno:
        napake.append((leto, zadnja_napaka))
        print(f"PREVERITI: {leto}")

print("\nUspešno pridobljenih let:", len(rezultati))
print("Leta za dodatno preverjanje:", napake)

OK: 2004 (IC2004), 6804 univerz
OK: 2005 (IC2005), 6844 univerz
OK: 2006 (IC2006), 6891 univerz
OK: 2007 (IC2007), 6902 univerz
OK: 2008 (IC2008), 7034 univerz
OK: 2009 (IC2009), 7135 univerz
OK: 2010 (IC2010), 7367 univerz
OK: 2011 (IC2011), 7591 univerz
OK: 2012 (IC2012), 7651 univerz
OK: 2013 (IC2013), 7614 univerz
OK: 2014 (ADM2014), 2236 univerz
OK: 2015 (ADM2015), 2210 univerz
OK: 2016 (ADM2016), 2068 univerz
OK: 2017 (ADM2017), 2075 univerz
OK: 2018 (ADM2018), 2038 univerz
OK: 2019 (ADM2019), 2011 univerz
OK: 2020 (ADM2020), 1989 univerz
OK: 2021 (ADM2021), 1981 univerz
OK: 2022 (ADM2022), 1991 univerz
OK: 2023 (ADM2023), 1972 univerz
OK: 2024 (ADM2024), 1956 univerz

Uspešno pridobljenih let: 21
Leta za dodatno preverjanje: []


In [ ]:
rv1 = pd.concat(rezultati, ignore_index=True) #zdruzevanje podatkov v 1 tabelo

print("Število zapisov:", len(rv1))
print("Število let:", rv1["leto"].nunique())
print("Število univerz:", rv1["UNITID"].nunique())

#preverjanje števila institucij po letih
print(rv1.groupby("leto")["UNITID"].nunique())

kazalniki = ["APPLCN", "ADMSSN", "ENRLT"]

for stolpec in kazalniki:
    stevilske = pd.to_numeric(rv1[stolpec], errors="coerce")

    #manjkajoči podatki in zapisi, ki niso prazni, vendar niso številke.
    maska = rv1[stolpec].notna() & stevilske.isna()

    print(f"\n{stolpec} – neštevilske oznake:")
    print(rv1.loc[maska, stolpec].value_counts())

    print("Že manjkajoče vrednosti:",
          rv1[stolpec].isna().sum())


Število zapisov: 94360
Število let: 21
Število univerz: 9061
leto
2004    6804
2005    6844
2006    6891
2007    6902
2008    7034
2009    7135
2010    7367
2011    7591
2012    7651
2013    7614
2014    2236
2015    2210
2016    2068
2017    2075
2018    2038
2019    2011
2020    1989
2021    1981
2022    1991
2023    1972
2024    1956
Name: UNITID, dtype: int64

APPLCN – neštevilske oznake:
APPLCN
.    45362
Name: count, dtype: int64
Že manjkajoče vrednosti: 0

ADMSSN – neštevilske oznake:
ADMSSN
.    45362
Name: count, dtype: int64
Že manjkajoče vrednosti: 236

ENRLT – neštevilske oznake:
ENRLT
.    45315
Name: count, dtype: int64
Že manjkajoče vrednosti: 283


In [ ]:

#neštevilske oznake (npr. .) pretvorim v NaN

for stolpec in kazalniki:
    rv1[stolpec] = pd.to_numeric(
        rv1[stolpec],
        errors="coerce"
    )

print("\nManjkajoče vrednosti po pretvorbi:")
print(rv1[kazalniki].isna().sum())

print("\nVrstice z vsaj enim manjkajočim podatkom:",
      rv1[kazalniki].isna().any(axis=1).sum())

print("Vrstice z vsemi tremi podatki:",
      rv1[kazalniki].notna().all(axis=1).sum())



Manjkajoče vrednosti po pretvorbi:
APPLCN    45362
ADMSSN    45598
ENRLT     45598
dtype: int64

Vrstice z vsaj enim manjkajočim podatkom: 45691
Vrstice z vsemi tremi podatki: 48669


In [ ]:
from google.colab import files
rv1.to_csv(
    "IPEDS_RV1.csv", #shranjevanje v csv
    index=False
)

files.download("IPEDS_RV1.csv") #prenos

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RV2: kako se je spreminjala stopnja zaključevanja študija?
(skozi leta in tudi skozi univerze)

priprava podatkov za analizo stopnje zaključevanja študija v 150 % predvidenega časa.

vir podatkov: IPEDS Graduation Rates (GR)

obdobje: 2008 - 2024

Opis stolpcev:

- GRTYPE - Vrsta podatka o kohorti oziroma zaključevanju študija (vrednosti: 2 - prilagojena začetna kohorta, 3 - študenti, ki so zaključili študij v 150 % predvidenega časa)

- CHRTSTAT - Oznaka statusa kohorte oziroma študentov (vrednosti: 12 - prilagojena kohorta, 13 - zaključevanje študija v 150 % predvidenega časa)

- SECTION - Oznaka dela oziroma razdelka podatkov o zaključevanju študija (V zbirki je ohranjena vrednost 1)

- COHORT - Oznaka vrste kohorte (V zbirki je ohranjena vrednost 1)

- GRTOTLT - Skupno število študentov v posamezni kategoriji, določeni z oznakami GRTYPE, CHRTSTAT, SECTION in COHORT. Podatek je izražen kot število oseb.

In [ ]:

vsi_gr = []
napake_gr = []

for leto in range(2008, 2025):
    try:
        df = prenesi_datoteko(f"GR{leto}")

        #preverjanje zahtevanih stolpcev
        potrebni = [
            "UNITID", "GRTYPE", "CHRTSTAT",
            "SECTION", "COHORT", "GRTOTLT"
        ]

        if not all(c in df.columns for c in potrebni):
            raise ValueError("Manjkajo zahtevani stolpci")

        df = df[potrebni].copy()

        df["leto"] = leto
        df["vir"] = f"GR{leto}"

        vsi_gr.append(df)

        print(f"OK: {leto}, {len(df)} zapisov")

    except Exception as e:
        napake_gr.append((leto, str(e)))
        print(f"PREVERITI: {leto} - {e}")

print("\nUspesno pridobljenih let:", len(vsi_gr))
print("Napake:", napake_gr)


OK: 2008, 36479 zapisov
OK: 2009, 42351 zapisov
OK: 2010, 43199 zapisov
OK: 2011, 55346 zapisov
OK: 2012, 56221 zapisov
OK: 2013, 56741 zapisov
OK: 2014, 57362 zapisov
OK: 2015, 57333 zapisov
OK: 2016, 55742 zapisov
OK: 2017, 54714 zapisov
OK: 2018, 51868 zapisov
OK: 2019, 50447 zapisov
OK: 2020, 50098 zapisov
OK: 2021, 49539 zapisov
OK: 2022, 48151 zapisov
OK: 2023, 51368 zapisov
OK: 2024, 51011 zapisov

Uspesno pridobljenih let: 17
Napake: []


In [ ]:
#zdruzevanje podatkov vseh let
rv2 = pd.concat(vsi_gr, ignore_index=True)

#preverjanje nestevilskih oznak pred pretvorbo
stevilske = pd.to_numeric(rv2["GRTOTLT"], errors="coerce")

maska = rv2["GRTOTLT"].notna() & stevilske.isna()

print("\nNestevilske oznake v GRTOTLT:")
print(rv2.loc[maska, "GRTOTLT"].value_counts())

print("\nManjkajoce vrednosti:")
print(rv2["GRTOTLT"].isna().sum())



Nestevilske oznake v GRTOTLT:
Series([], Name: count, dtype: int64)

Manjkajoce vrednosti:
0


In [ ]:
print("Stevilo zapisov:", len(rv2))
print("Stevilo let:", rv2["leto"].nunique())
print("Stevilo institucij:", rv2["UNITID"].nunique())

print("\nStevilo institucij po letih:")
print(rv2.groupby("leto")["UNITID"].nunique())

print("\nIvy League po letih:")
print(
    rv2[rv2["UNITID"].isin(ivy)]
    .groupby("leto")["UNITID"]
    .nunique()
)

rv2.to_csv(
    "IPEDS_RV2.csv",
    index=False
)

files.download("IPEDS_RV2.csv")

Stevilo zapisov: 867970
Stevilo let: 17
Stevilo institucij: 5769

Stevilo institucij po letih:
leto
2008    4406
2009    4440
2010    4519
2011    4573
2012    4576
2013    4598
2014    4591
2015    4533
2016    4359
2017    4260
2018    4016
2019    3882
2020    3852
2021    3800
2022    3784
2023    3726
2024    3684
Name: UNITID, dtype: int64

Ivy League po letih:
leto
2008    8
2009    8
2010    8
2011    8
2012    8
2013    8
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    8
2021    8
2022    8
2023    8
2024    8
Name: UNITID, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RV4: kako se je spreminjalo število in delež diplomantov računalništva ter sorodnih programov?

podatkovni vir: IPEDS Completions (C)
obdobje: 2004 - 2024

stolpci:
- CIPCODE - koda študijskega področja/programa
- AWLEVEL - raven pridobljene izobrazbe oz. vrste pridobljenega naziva
- MAJORNUM - oznaka če je ta program študentova prva ali druga glavna študijska smer (ker lahko študirajo več področij)
- CTOTAL - skupno število podeljenih diplom oz. nazivov (od 2008)
- CRACE24 - skupno število podeljenih diplom oz. nazivov (od 2004 do 2007)

In [ ]:
vsi_c = []
napake_c = []

for leto in range(2004, 2025):
    try:
        df = prenesi_datoteko(f"C{leto}_A")

        #odstranitev presledkov, drugih znakov
        df.columns = (
            df.columns
            .str.replace("ï»¿", "", regex=False)
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
        )

        #preverjanje zahtevanih stolpcev
        potrebni = [
            "UNITID", "CIPCODE", "AWLEVEL", "MAJORNUM"
        ]

        if not all(c in df.columns for c in potrebni):
            raise ValueError("Manjkajo zahtevani stolpci")

        #preverjanje stolpca za skupno stevilo diplom - v razlicnih letih je drugace poimenovan
        if "CTOTALT" in df.columns:
            stevec = "CTOTALT"
        elif "CRACE24" in df.columns:
            stevec = "CRACE24"
        else:
            raise ValueError("Ni ustreznega stolpca za diplome")

        df = df[potrebni + [stevec]].copy()
        df["leto"] = leto
        df["vir"] = f"C{leto}_A"

        vsi_c.append(df)

        print(f"OK: {leto}, {len(df)} zapisov ({stevec})")

    except Exception as e:
        napake_c.append((leto, str(e)))
        print(f"PREVERITI: {leto} - {e}")

print("\nUspesno pridobljenih let:", len(vsi_c))
print("Napake:", napake_c)

OK: 2004, 219682 zapisov (CRACE24)
OK: 2005, 230135 zapisov (CRACE24)
OK: 2006, 238406 zapisov (CRACE24)
OK: 2007, 240507 zapisov (CRACE24)
OK: 2008, 247075 zapisov (CTOTALT)
OK: 2009, 255884 zapisov (CTOTALT)
OK: 2010, 265604 zapisov (CTOTALT)
OK: 2011, 265808 zapisov (CTOTALT)
OK: 2012, 270166 zapisov (CTOTALT)
OK: 2013, 284907 zapisov (CTOTALT)
OK: 2014, 293259 zapisov (CTOTALT)
OK: 2015, 300222 zapisov (CTOTALT)
OK: 2016, 301984 zapisov (CTOTALT)
OK: 2017, 308943 zapisov (CTOTALT)
OK: 2018, 290119 zapisov (CTOTALT)
OK: 2019, 291052 zapisov (CTOTALT)
OK: 2020, 289792 zapisov (CTOTALT)
OK: 2021, 296343 zapisov (CTOTALT)
OK: 2022, 300877 zapisov (CTOTALT)
OK: 2023, 303292 zapisov (CTOTALT)
OK: 2024, 307707 zapisov (CTOTALT)

Uspesno pridobljenih let: 21
Napake: []


In [ ]:
rv4 = pd.concat(vsi_c, ignore_index=True)

#preverjanje nestevilskih oznak pred pretvorbo
kazalniki = ["CTOTALT", "CRACE24"]

for stolpec in kazalniki:
    if stolpec in rv4.columns:

        stevilske = pd.to_numeric(
            rv4[stolpec], errors="coerce"
        )

        maska = (
            rv4[stolpec].notna() &
            stevilske.isna()
        )

        print(f"\nNestevilske oznake v {stolpec}:")
        print(rv4.loc[maska, stolpec].value_counts())

#preverjanje manjkajocih vrednosti
print("\nManjkajoce vrednosti:")
print(rv4.isna().sum())


Nestevilske oznake v CTOTALT:
Series([], Name: count, dtype: int64)

Nestevilske oznake v CRACE24:
Series([], Name: count, dtype: int64)

Manjkajoce vrednosti:
UNITID            0
CIPCODE           0
AWLEVEL           0
MAJORNUM          0
CRACE24     4873034
leto              0
vir               0
CTOTALT      928730
dtype: int64


za skupno število diplom sta stolpca:
- CTOTAL (iz novejših datotek - od 2008)
- CRACE24 (iz starejših datotek - od 2004 do 2007)

V letih, ko posamezni stolpec ne obstaja so vrednosti NaN

In [ ]:
print("Stevilo zapisov:", len(rv4))
print("Stevilo let:", rv4["leto"].nunique())
print("Stevilo institucij:", rv4["UNITID"].nunique())

print("\nStevilo institucij po letih:")
print(rv4.groupby("leto")["UNITID"].nunique())

print("\nIvy League po letih:")
print(
    rv4[rv4["UNITID"].isin(ivy)]
    .groupby("leto")["UNITID"]
    .nunique()
)

print("\nRazpolozljivost stolpcev po letih:")
print(
    rv4.groupby("leto")[["CTOTALT", "CRACE24"]]
    .count()
)

#shranjevanje
rv4.to_csv(
    "IPEDS_RV4.csv",
    index=False
)

files.download("IPEDS_RV4.csv")

Stevilo zapisov: 5801764
Stevilo let: 21
Stevilo institucij: 9746

Stevilo institucij po letih:
leto
2004    6636
2005    6721
2006    6772
2007    6785
2008    6913
2009    7030
2010    7269
2011    7489
2012    7552
2013    7513
2014    7430
2015    7322
2016    6950
2017    6788
2018    6571
2019    6269
2020    6173
2021    6089
2022    6042
2023    5960
2024    5869
Name: UNITID, dtype: int64

Ivy League po letih:
leto
2004    8
2005    8
2006    8
2007    8
2008    8
2009    8
2010    8
2011    8
2012    8
2013    8
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    8
2021    8
2022    8
2023    8
2024    8
Name: UNITID, dtype: int64

Razpolozljivost stolpcev po letih:
      CTOTALT  CRACE24
leto                  
2004        0   219682
2005        0   230135
2006        0   238406
2007        0   240507
2008   247075        0
2009   255884        0
2010   265604        0
2011   265808        0
2012   270166        0
2013   284907        0
2014   293259        0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PAZI! - omejitev:
V podatkih se pojavlja vrednost 99.0 (predstavlja skupni seštevek?)

RV3: kako so se spreminjali stroški študija na ameriških univerzah?

podatkovni vir: IPEDS Institutional Characteristics (IC)
obdobje: 2004 - 2023

spremenljivke:
- CHG1AY0–3 - Objavljena šolnina in obvezne pristojbine za študente iz istega okrožja (in-district).
- CHG2AY0–3 - Objavljena šolnina in obvezne pristojbine za študente iz iste zvezne države (in-state).
- CHG3AY0–3 - Objavljena šolnina in obvezne pristojbine za študente iz drugih zveznih držav (out-of-state).

Končnice AY0, AY1, AY2 in AY3 označujejo različna študijska leta. (AY3 - študijsko leto, ki se začne v letu iz imena datoteke)

In [ ]:
import requests
import zipfile
import io
import pandas as pd

def prenesi_datoteko(ime):
    naslovi = [
        f"https://nces.ed.gov/ipeds/datacenter/data/{ime}.zip",
        f"https://nces.ed.gov/ipeds/complete-data-files/{ime}.zip"
    ]

    zadnja_napaka = None

    for url in naslovi:
        try:
            odgovor = requests.get(url, timeout=60)
            odgovor.raise_for_status()

            with zipfile.ZipFile(io.BytesIO(odgovor.content)) as z:
                csv_ime = next(
                    f for f in z.namelist()
                    if f.lower().endswith(".csv")
                )

                with z.open(csv_ime) as f:
                    df = pd.read_csv(
                        f,
                        encoding="latin1",
                        index_col=False,
                        low_memory=False
                    )

            #ureditev imen stolpcev
            df.columns = (
                df.columns
                .str.replace("\ufeff", "", regex=False)
                .str.replace("ï»¿", "", regex=False)
                .str.strip()
            )

            #pretvorba identifikatorja institucije v stevilo
            if "UNITID" in df.columns:
                df["UNITID"] = pd.to_numeric(
                    df["UNITID"],
                    errors="coerce"
                )

            return df

        except Exception as e:
            zadnja_napaka = e

    raise RuntimeError(
        f"Prenos datoteke {ime} ni uspel: {zadnja_napaka}"
    )

In [ ]:
import pandas as pd

vsi_rv3 = []
napake_rv3 = []

for leto in range(2004, 2024):
    ime = f"IC{leto}_AY"

    try:
        df = prenesi_datoteko(ime)

        if "UNITID" not in df.columns:
            raise ValueError("Manjka stolpec UNITID")

        #izbor razpolozljivih izvornih stroskovnih stolpcev
        stroskovni = [
            c for c in df.columns
            if c.startswith(("CHG1AY", "CHG2AY", "CHG3AY"))
        ]

        if not stroskovni:
            raise ValueError("Stroskovni stolpci niso najdeni")

        df = df[["UNITID"] + stroskovni].copy()

        df["leto"] = leto
        df["vir"] = ime

        vsi_rv3.append(df)

        print(
            f"OK: {leto}, {len(df)} zapisov, "
            f"stolpci: {stroskovni}"
        )

    except Exception as e:
        napake_rv3.append((leto, str(e)))
        print(f"PREVERITI: {leto} - {e}")

print("\nUspesno pridobljenih let:", len(vsi_rv3))
print("Napake:", napake_rv3)

OK: 2004, 4631 zapisov, stolpci: ['CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK: 2005, 4568 zapisov, stolpci: ['CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK: 2006, 4573 zapisov, stolpci: ['CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK: 2007, 4563 zapisov, stolpci: ['CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK: 2008, 4603 zapisov, stolpci: ['CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK: 2009, 4661 zapisov, stolpci: ['CHG1AY0', 'CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY0', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY0', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK: 2010, 4678 zapisov, stolpci: ['CHG1AY0', 'CHG1AY1', 'CHG1AY2', 'CHG1AY3', 'CHG2AY0', 'CHG2AY1', 'CHG2AY2', 'CHG2AY3', 'CHG3AY0', 'CHG3AY1', 'CHG3AY2', 'CHG3AY3']
OK:

In [ ]:
rv3 = pd.concat(vsi_rv3, ignore_index=True)

#izbor stroskovnih stolpcev
stroskovni = [
    c for c in rv3.columns
    if c.startswith(("CHG1AY", "CHG2AY", "CHG3AY"))
]

#preverjanje nestevilskih oznak in pretvorba v NaN
for stolpec in stroskovni:

    stevilske = pd.to_numeric(
        rv3[stolpec],
        errors="coerce"
    )

    maska = rv3[stolpec].notna() & stevilske.isna()

    print(f"\n{stolpec} - nestevilske oznake:")
    print(rv3.loc[maska, stolpec].value_counts())

    rv3[stolpec] = stevilske

print("\nManjkajoce vrednosti po pretvorbi:")
print(rv3[stroskovni].isna().sum())


CHG1AY1 - nestevilske oznake:
CHG1AY1
.    10299
Name: count, dtype: int64

CHG1AY2 - nestevilske oznake:
CHG1AY2
.    9751
Name: count, dtype: int64

CHG1AY3 - nestevilske oznake:
CHG1AY3
.    9371
Name: count, dtype: int64

CHG2AY1 - nestevilske oznake:
CHG2AY1
.    10301
Name: count, dtype: int64

CHG2AY2 - nestevilske oznake:
CHG2AY2
.    9751
Name: count, dtype: int64

CHG2AY3 - nestevilske oznake:
CHG2AY3
.    9371
Name: count, dtype: int64

CHG3AY1 - nestevilske oznake:
CHG3AY1
.    10306
Name: count, dtype: int64

CHG3AY2 - nestevilske oznake:
CHG3AY2
.    9755
Name: count, dtype: int64

CHG3AY3 - nestevilske oznake:
CHG3AY3
.    9371
Name: count, dtype: int64

CHG1AY0 - nestevilske oznake:
CHG1AY0
.    8127
Name: count, dtype: int64

CHG2AY0 - nestevilske oznake:
CHG2AY0
.    8127
Name: count, dtype: int64

CHG3AY0 - nestevilske oznake:
CHG3AY0
.    8127
Name: count, dtype: int64

Manjkajoce vrednosti po pretvorbi:
CHG1AY1    10299
CHG1AY2     9751
CHG1AY3     9371
CHG2AY1   

In [ ]:
print("Stevilo zapisov:", len(rv3))
print("Stevilo let:", rv3["leto"].nunique())
print("Stevilo institucij:", rv3["UNITID"].nunique())

print("\nStevilo institucij po letih:")
print(rv3.groupby("leto")["UNITID"].nunique())

print("\nIvy League po letih:")
print(
    rv3[rv3["UNITID"].isin(ivy)]
    .groupby("leto")["UNITID"]
    .nunique()
)

#preverjanje podvojenih zapisov
print("\nPodvojeni UNITID-leto:")
print(
    rv3.duplicated(
        subset=["UNITID", "leto"]
    ).sum()
)

print("\nRazpolozljivost stroskovnih podatkov po letih:")
display(
    rv3.groupby("leto")[stroskovni].count()
)

#prenos
if not napake_rv3:
    rv3.to_csv(
        "IPEDS_RV3.csv",
        index=False
    )

    files.download("IPEDS_RV3.csv")
else:
    print("Izvoz ni izveden zaradi napak pri prenosu.")

Stevilo zapisov: 88205
Stevilo let: 20
Stevilo institucij: 6096

Stevilo institucij po letih:
leto
2004    4631
2005    4568
2006    4573
2007    4563
2008    4603
2009    4661
2010    4678
2011    4793
2012    4806
2013    4783
2014    4660
2015    4607
2016    4397
2017    4281
2018    4125
2019    3993
2020    3911
2021    3871
2022    3876
2023    3825
Name: UNITID, dtype: int64

Ivy League po letih:
leto
2004    8
2005    8
2006    8
2007    8
2008    8
2009    8
2010    8
2011    8
2012    8
2013    8
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    8
2021    8
2022    8
2023    8
Name: UNITID, dtype: int64

Podvojeni UNITID-leto:
0

Razpolozljivost stroskovnih podatkov po letih:


,CHG1AY1,CHG1AY2,CHG1AY3,CHG2AY1,CHG2AY2,CHG2AY3,CHG3AY1,CHG3AY2,CHG3AY3,CHG1AY0,CHG2AY0,CHG3AY0
leto,,,,,,,,,,,,
2004,3996,4109,4173,3994,4109,4173,3990,4106,4173,0,0,0
2005,4075,4102,4119,4075,4102,4119,4074,4101,4119,0,0,0
2006,4106,4121,4133,4106,4121,4133,4106,4121,4133,0,0,0
2007,4089,4103,4131,4089,4103,4131,4089,4103,4131,0,0,0
2008,4132,4143,4161,4132,4143,4161,4132,4143,4161,0,0,0
2009,4127,4167,4188,4127,4167,4188,4127,4167,4188,4096,4096,4096
2010,4126,4178,4200,4126,4178,4200,4126,4178,4200,4083,4083,4083
2011,4216,4267,4292,4216,4267,4292,4216,4267,4292,4162,4162,4162
2012,4192,4242,4306,4192,4242,4306,4192,4242,4306,4147,4147,4147


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>